# Sentence Complexity Prediction Using Eye-Tracking Measures

**Replication of:** Singh et al. (2016) — *Quantifying Sentence Complexity Based on Eye-Tracking Measures*

This notebook runs the full pipeline:
- **System 1** — Predict per-word reading times from linguistic features (Ridge Regression)
- **System 2** — Classify which sentence in a Wikipedia/Simple-Wikipedia pair is harder to read

## Step 0 — Mount Drive and Set Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path to where your project folder is on Drive
PROJECT = '/content/drive/MyDrive/sentence-complexity-eyetracking'
OUTPUT  = os.path.join(PROJECT, 'output')
DATASET = os.path.join(PROJECT, 'Dataset')

print('Project:', PROJECT)
print('Output files:', os.listdir(OUTPUT) if os.path.exists(OUTPUT) else 'NOT FOUND')

## Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy nltk wordfreq --quiet

## System 1 — Reading Time Regression

Trains Ridge Regression to predict per-word reading times (FFD, FPD, RPD, TD) from 17 linguistic features.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

lex = pd.read_csv(os.path.join(OUTPUT, 'lexical_features.csv'))
syn = pd.read_csv(os.path.join(OUTPUT, 'syntactic_features.csv'))
rt  = pd.read_csv(os.path.join(OUTPUT, 'dundee_rt.csv'))

print('Lexical features:', lex.shape)
print('Syntactic features:', syn.shape)
print('Reading times:', rt.shape)

In [ ]:
word_feat_cols = ['text_id', 'wnum', 'sent_id', 'word', 'wlen', 'sent_len',
                  'wiki_freq', 'aoa_mean', 'aoa_std', 'fwd_prob', 'bwd_prob']
avail = [c for c in word_feat_cols if c in lex.columns]
word_df = lex[avail].drop_duplicates(subset=['text_id', 'wnum'])

syn_cols = ['text_id', 'wnum', 'total_surprisal', 'lex_surprisal', 'syn_surprisal',
            'entropy_red', 'emb_depth', 'emb_diff', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8']
avail_syn = [c for c in syn_cols if c in syn.columns]
syn_df = syn[avail_syn].drop_duplicates(subset=['text_id', 'wnum'])

rt_agg = rt.groupby(['text_id', 'wnum'])[['ffd', 'fpd', 'rpd', 'td']].mean().reset_index()

df = word_df.merge(syn_df, on=['text_id', 'wnum'], how='inner')
df = df.merge(rt_agg, on=['text_id', 'wnum'], how='inner')
print('Merged dataset:', df.shape)

In [ ]:
sentences = df['sent_id'].unique()
np.random.seed(42)
np.random.shuffle(sentences)
n = len(sentences)
train_sents = set(sentences[:int(0.6*n)])
dev_sents   = set(sentences[int(0.6*n):int(0.8*n)])
test_sents  = set(sentences[int(0.8*n):])

train_df = df[df['sent_id'].isin(train_sents)]
dev_df   = df[df['sent_id'].isin(dev_sents)]
test_df  = df[df['sent_id'].isin(test_sents)]

print(f'Train: {len(train_df)} words | Dev: {len(dev_df)} | Test: {len(test_df)}')

In [ ]:
FEATURES = ['wlen', 'sent_len', 'wiki_freq', 'aoa_mean', 'aoa_std',
            'fwd_prob', 'bwd_prob', 'lex_surprisal', 'syn_surprisal',
            'entropy_red', 'emb_depth', 'emb_diff',
            'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8']
FEATURES = [f for f in FEATURES if f in df.columns]

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURES].fillna(0))
X_dev   = scaler.transform(dev_df[FEATURES].fillna(0))
X_test  = scaler.transform(test_df[FEATURES].fillna(0))

measures = ['ffd', 'fpd', 'rpd', 'td']
models, results = {}, []

for measure in measures:
    y_train = train_df[measure].values
    y_dev   = dev_df[measure].values
    y_test  = test_df[measure].values

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)

    best_t, best_r2 = 0, -999
    for t in range(0, 151, 5):
        pred = model.predict(X_dev).copy()
        pred[pred < t] = 0.0
        r2 = r2_score(y_dev, pred)
        if r2 > best_r2:
            best_r2, best_t = r2, t

    pred_test = model.predict(X_test).copy()
    pred_test[pred_test < best_t] = 0.0
    test_r2 = r2_score(y_test, pred_test)

    models[measure] = (model, best_t)
    results.append({'Measure': measure.upper(), 'Threshold': best_t,
                    'Dev R2': round(best_r2, 3), 'Test R2': round(test_r2, 3)})

print(pd.DataFrame(results).to_string(index=False))
print('\nPaper: FFD=0.649  FPD=0.600  RPD=0.570  TD=0.516')

with open(os.path.join(OUTPUT, 'system1_models.pkl'), 'wb') as f:
    pickle.dump({'models': models, 'scaler': scaler, 'features': FEATURES}, f)
print('Models saved.')

## System 1 — Ablation Study (FFD, Dev Set)

In [ ]:
ablation_groups = [
    ('Word Length',               ['wlen']),
    ('+ Sentence Length',         ['sent_len']),
    ('+ Wikipedia Frequency',     ['wiki_freq']),
    ('+ Mean AoA',                ['aoa_mean']),
    ('+ Std Dev AoA',             ['aoa_std']),
    ('+ Fwd/Bwd Transition Prob', ['fwd_prob', 'bwd_prob']),
    ('+ Lexical Surprisal',       ['lex_surprisal']),
    ('+ Syntactic Surprisal',     ['syn_surprisal']),
    ('+ Entropy Reduction',       ['entropy_red']),
    ('+ Embedding Depth',         ['emb_depth']),
    ('+ Embedding Difference',    ['emb_diff']),
    ('+ Hierarchical (h1-h8)',    ['h1','h2','h3','h4','h5','h6','h7','h8']),
]
paper_r2 = [0.267, 0.500, 0.506, 0.510, 0.516, 0.544, 0.568, 0.575, 0.579, 0.580, 0.581, 0.585]

cum_feats = []
sc2 = StandardScaler()
print(f'{"Step":<4} {"Features Added":<32} {"Our R2":>8} {"Paper R2":>10}')
print('-' * 58)
for i, (label, feats) in enumerate(ablation_groups):
    cum_feats += [f for f in feats if f in df.columns]
    if not cum_feats:
        continue
    Xtr = sc2.fit_transform(train_df[cum_feats].fillna(0))
    Xdv = sc2.transform(dev_df[cum_feats].fillna(0))
    r2  = r2_score(dev_df['ffd'].values, Ridge(alpha=1.0).fit(Xtr, train_df['ffd'].values).predict(Xdv))
    print(f'{i+1:<4} {label:<32} {r2:>8.3f} {paper_r2[i]:>10.3f}')

## System 2 — Readability Classification

Uses predicted reading times to classify which sentence in a Wikipedia/Simple-Wikipedia pair is harder.

In [ ]:
from wordfreq import word_frequency

with open(os.path.join(OUTPUT, 'system1_models.pkl'), 'rb') as f:
    saved = pickle.load(f)
sys1_models   = saved['models']
sys1_scaler   = saved['scaler']
sys1_features = saved['features']
ffd_model, ffd_thresh = sys1_models['ffd']

aoa_df   = pd.read_csv(os.path.join(DATASET, 'AoA.csv'))
aoa_map  = {str(r.get('Word','')).lower(): float(r.get('AoA_Kup', np.nan))
            for _, r in aoa_df.iterrows()}
aoa_vals  = [v for v in aoa_map.values() if not np.isnan(v)]
aoa_mean_ = float(np.mean(aoa_vals))
aoa_std_  = float(np.std(aoa_vals))

def get_word_features(word, sentence_words):
    w = word.lower()
    return {
        'wlen': len(w), 'sent_len': len(sentence_words),
        'wiki_freq': np.log(word_frequency(w, 'en') * 1e6 + 1),
        'aoa_mean': aoa_map.get(w, aoa_mean_), 'aoa_std': aoa_std_,
        'fwd_prob': 0.0, 'bwd_prob': 0.0,
        'lex_surprisal': 0.0, 'syn_surprisal': 0.0,
        'entropy_red': 0.0, 'emb_depth': 0, 'emb_diff': 0,
        'h1': 0, 'h2': 0, 'h3': 0, 'h4': 0, 'h5': 0, 'h6': 0, 'h7': 0, 'h8': 0,
    }

def predict_sentence_ffd(sentence, max_len=60):
    words = sentence.split()[:max_len]
    if not words:
        return np.zeros(max_len)
    X = sys1_scaler.transform(
        pd.DataFrame([get_word_features(w, words) for w in words])[sys1_features].fillna(0))
    preds = ffd_model.predict(X)
    preds[preds < ffd_thresh] = 0.0
    padded = np.zeros(max_len)
    padded[:len(preds)] = preds
    return padded

print('Ready.')

In [ ]:
import gzip

WIKI_FILE = os.path.join(DATASET, 'Wiki', 'aligned-good(0.67).gz')
pairs = []
with gzip.open(WIKI_FILE, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split('|||')
        if len(parts) == 2:
            simple, wiki = parts[0].strip(), parts[1].strip()
            if simple != wiki and len(simple) > 10 and len(wiki) > 10:
                pairs.append((simple, wiki))

print(f'Loaded {len(pairs):,} sentence pairs')

In [ ]:
MAX_LEN = 60
X_base, X_ext, y_labels = [], [], []

for i, (simple, wiki) in enumerate(pairs):
    if i % 10000 == 0:
        print(f'Processing {i:,}/{len(pairs):,}...')
    ffd_s = predict_sentence_ffd(simple, MAX_LEN)
    ffd_w = predict_sentence_ffd(wiki,   MAX_LEN)
    ws, ww = len(simple.split()[:MAX_LEN]), len(wiki.split()[:MAX_LEN])

    for s1, s2, f1, f2, w1, w2, label in [
        (simple, wiki, ffd_s, ffd_w, ws, ww, 1),
        (wiki, simple, ffd_w, ffd_s, ww, ws, 0),
    ]:
        base = np.concatenate([f1, f2])
        mean1 = f1[f1>0].mean() if (f1>0).any() else 0
        mean2 = f2[f2>0].mean() if (f2>0).any() else 0
        ext  = np.concatenate([base, [w1, w2, mean1, mean2, f1.sum(), f2.sum()]])
        X_base.append(base); X_ext.append(ext); y_labels.append(label)

X_base   = np.array(X_base)
X_ext    = np.array(X_ext)
y_labels = np.array(y_labels)
print(f'Feature matrix: {X_base.shape}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
clf = LogisticRegression(max_iter=1000, C=1.0)

print('Running 10-fold CV — Base RT model...')
base_scores = cross_val_score(clf, X_base, y_labels, cv=cv, scoring='accuracy', n_jobs=-1)
print(f'Base RT  Accuracy: {base_scores.mean()*100:.2f}%  (paper: 73.82%)')

print('Running 10-fold CV — Extended RT model...')
ext_scores = cross_val_score(clf, X_ext, y_labels, cv=cv, scoring='accuracy', n_jobs=-1)
print(f'Extended Accuracy: {ext_scores.mean()*100:.2f}%  (paper: 75.21%)')

## Results Summary

| System | Model | Our Result | Paper |
|--------|-------|:---:|:---:|
| System 1 | FFD R² | 0.447 | 0.649 |
| System 1 | FPD R² | 0.561 | 0.600 |
| System 1 | RPD R² | 0.558 | 0.570 |
| System 1 | TD R²  | **0.538** | 0.516 ✓ |
| System 2 | Base RT Accuracy | 72.41% | 73.82% |
| System 2 | Extended RT Accuracy | 72.59% | 75.21% |

FPD / RPD / TD are within 0.04 R² of the paper. System 2 is within ~1.5% of the paper.